In [1]:
import pandas as pd
from pathlib import Path

df = pd.read_csv("../data/emi_prediction_dataset.csv")

# 1. Inspect data types
print("=== DATA TYPES ===")
print(df.dtypes.to_string())

# 2. Inspect missing-value percentages
print("\n=== MISSING VALUE PERCENTAGES ===")
missing_pct = df.isnull().mean().mul(100)
print(missing_pct[missing_pct > 0].round(2).to_string())

# 3. Inspect classification target
print("\n=== EMI ELIGIBILITY DISTRIBUTION ===")
print(df["emi_eligibility"].value_counts(dropna=False))

print("\n=== EMI ELIGIBILITY PERCENTAGES ===")
print(
    df["emi_eligibility"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

# 4. Inspect regression target
print("\n=== MAX MONTHLY EMI SUMMARY ===")
print(df["max_monthly_emi"].describe())

# 5. Inspect EMI scenarios
print("\n=== EMI SCENARIO DISTRIBUTION ===")
print(df["emi_scenario"].value_counts(dropna=False))


C:\Users\Rahul Kiran Hosuru\AppData\Local\Temp\ipykernel_14256\4199884105.py:4: DtypeWarning: Columns (0: age) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/emi_prediction_dataset.csv")


=== DATA TYPES ===
age                        object
gender                        str
marital_status                str
education                     str
monthly_salary                str
employment_type               str
years_of_employment       float64
company_type                  str
house_type                    str
monthly_rent              float64
family_size                 int64
dependents                  int64
school_fees               float64
college_fees              float64
travel_expenses           float64
groceries_utilities       float64
other_monthly_expenses    float64
existing_loans                str
current_emi_amount        float64
credit_score              float64
bank_balance                  str
emergency_fund            float64
emi_scenario                  str
requested_amount          float64
requested_tenure            int64
emi_eligibility               str
max_monthly_emi           float64

=== MISSING VALUE PERCENTAGES ===
education         0.59
month

In [2]:

# Inspect values in columns that should be numeric
columns_to_check = [
    "age",
    "monthly_salary",
    "bank_balance"
]

for col in columns_to_check:
    print(f"\n--- {col} ---")
    print("Data type:", df[col].dtype)
    print("Sample values:")
    print(df[col].dropna().head(10).tolist())
    print("Unique sample formats:")
    print(df[col].dropna().astype(str).str.strip().head(10).tolist())



--- age ---
Data type: object
Sample values:
[38.0, 38.0, 38.0, 58.0, 48.0, 38.0, 58.0, 58.0, 58.0, 32.0]
Unique sample formats:
['38.0', '38.0', '38.0', '58.0', '48.0', '38.0', '58.0', '58.0', '58.0', '32.0']

--- monthly_salary ---
Data type: str
Sample values:
['82600.0', '21500.0', '86100.0', '66800.0', '57300.0', '38800.0', '27100.0', '392044.0', '47700.0', '129200.0']
Unique sample formats:
['82600.0', '21500.0', '86100.0', '66800.0', '57300.0', '38800.0', '27100.0', '392044.0', '47700.0', '129200.0']

--- bank_balance ---
Data type: str
Sample values:
['303200.0', '92500.0', '672100.0', '440900.0', '97300.0', '260800.0', '68000.0', '184600.0', '235600.0', '963200.0']
Unique sample formats:
['303200.0', '92500.0', '672100.0', '440900.0', '97300.0', '260800.0', '68000.0', '184600.0', '235600.0', '963200.0']


In [3]:
import pandas as pd

# Keep the original dataset unchanged
df_clean = df.copy()

# Columns that should contain numeric values
numeric_columns = [
    "age",
    "monthly_salary",
    "bank_balance"
]

for col in numeric_columns:
    # Convert invalid values to NaN instead of crashing
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

print("Conversion complete!\n")

print("Updated data types:")
print(df_clean[numeric_columns].dtypes)

print("\nMissing values after conversion:")
print(df_clean[numeric_columns].isna().sum())

print("\nSample converted values:")
display(df_clean[numeric_columns].head(10))


Conversion complete!

Updated data types:
age               float64
monthly_salary    float64
bank_balance      float64
dtype: object

Missing values after conversion:
age                  3
monthly_salary    1993
bank_balance      4392
dtype: int64

Sample converted values:


,age,monthly_salary,bank_balance
0,38.0,82600.0,303200.0
1,38.0,21500.0,92500.0
2,38.0,86100.0,672100.0
3,58.0,66800.0,440900.0
4,48.0,57300.0,97300.0
5,38.0,38800.0,260800.0
6,58.0,27100.0,68000.0
7,58.0,392044.0,184600.0
8,58.0,47700.0,235600.0
9,32.0,129200.0,963200.0


In [4]:

columns_to_check = ["age", "monthly_salary", "bank_balance"]

for col in columns_to_check:
    original_missing = df[col].isna().sum()
    converted_missing = df_clean[col].isna().sum()

    # Values that existed originally but could not be converted
    failed = df[col].notna() & df_clean[col].isna()

    print(f"\n--- {col} ---")
    print("Missing before conversion:", original_missing)
    print("Missing after conversion:", converted_missing)
    print("Failed conversions:", failed.sum())

    if failed.any():
        print("Examples of failed values:")
        print(df.loc[failed, col].astype(str).value_counts().head(10))



--- age ---
Missing before conversion: 0
Missing after conversion: 3
Failed conversions: 3
Examples of failed values:
age
58.0.0    1
38.0.0    1
32.0.0    1
Name: count, dtype: int64

--- monthly_salary ---
Missing before conversion: 0
Missing after conversion: 1993
Failed conversions: 1993
Examples of failed values:
monthly_salary
18000.0.0    34
20000.0.0    16
23400.0.0    11
14400.0.0    11
19500.0.0     9
26000.0.0     9
50500.0.0     9
58600.0.0     8
32200.0.0     8
48000.0.0     8
Name: count, dtype: int64

--- bank_balance ---
Missing before conversion: 2426
Missing after conversion: 4392
Failed conversions: 1966
Examples of failed values:
bank_balance
nan.0         14
258300.0.0     4
183300.0.0     4
144300.0.0     3
327400.0.0     3
101400.0.0     3
52300.0.0      3
64100.0.0      3
198500.0.0     3
61100.0.0      3
Name: count, dtype: int64


In [5]:

for col in ["age", "monthly_salary", "bank_balance"]:
    failed = df[col].notna() & df_clean[col].isna()

    print(f"\n--- {col}: malformed values ---")
    print(df.loc[failed, col].astype(str).value_counts().head(15))



--- age: malformed values ---
age
58.0.0    1
38.0.0    1
32.0.0    1
Name: count, dtype: int64

--- monthly_salary: malformed values ---
monthly_salary
18000.0.0    34
20000.0.0    16
23400.0.0    11
14400.0.0    11
19500.0.0     9
26000.0.0     9
50500.0.0     9
58600.0.0     8
32200.0.0     8
48000.0.0     8
15000.0.0     8
27000.0.0     7
52300.0.0     7
38800.0.0     7
45000.0.0     7
Name: count, dtype: int64

--- bank_balance: malformed values ---
bank_balance
nan.0         14
258300.0.0     4
183300.0.0     4
144300.0.0     3
327400.0.0     3
101400.0.0     3
52300.0.0      3
64100.0.0      3
198500.0.0     3
61100.0.0      3
254500.0.0     3
96800.0.0      3
84000.0.0      3
136800.0.0     3
269900.0.0     3
Name: count, dtype: int64


In [6]:

df_clean = df.copy()

for col in ["age", "monthly_salary", "bank_balance"]:
    cleaned_values = (
        df_clean[col]
        .astype("string")
        .str.strip()
        .str.replace(r"(?<=\.0)\.0$", "", regex=True)
    )

    df_clean[col] = pd.to_numeric(cleaned_values, errors="coerce")

print("Updated data types:")
print(df_clean[["age", "monthly_salary", "bank_balance"]].dtypes)

print("\nMissing values after correction:")
print(df_clean[["age", "monthly_salary", "bank_balance"]].isna().sum())


Updated data types:
age               Float64
monthly_salary    Float64
bank_balance      Float64
dtype: object

Missing values after correction:
age                  0
monthly_salary       3
bank_balance      2447
dtype: int64


In [7]:
for col in ["monthly_salary", "bank_balance"]:
    failed = df[col].notna() & df_clean[col].isna()

    print(f"\n--- {col} ---")
    print("Failed conversions:", failed.sum())

    if failed.any():
        print("Examples:")
        print(df.loc[failed, col].astype(str).value_counts().head(20))



--- monthly_salary ---
Failed conversions: 3
Examples:
monthly_salary
23400.0.0.0    1
81400.0.0.0    1
30400.0.0.0    1
Name: count, dtype: int64

--- bank_balance ---
Failed conversions: 21
Examples:
bank_balance
nan.0           14
316400.0.0.0     1
196900.0.0.0     1
380000.0.0.0     1
133100.0.0.0     1
89800.0.0.0      1
58700.0.0.0      1
444500.0.0.0     1
Name: count, dtype: int64


In [8]:

print("Age range:")
print(df_clean["age"].describe())

print("\nMonthly salary range:")
print(df_clean["monthly_salary"].describe())

print("\nBank balance range:")
print(df_clean["bank_balance"].describe())


Age range:
count    404800.0
mean     38.87586
std      9.303593
min          26.0
25%          32.0
50%          38.0
75%          48.0
max          59.0
Name: age, dtype: Float64

Monthly salary range:
count        404797.0
mean     59501.994511
std      43367.336847
min            3967.0
25%           35400.0
50%           51700.0
75%           73000.0
max          499970.0
Name: monthly_salary, dtype: Float64

Bank balance range:
count         402353.0
mean     241689.148086
std      183227.325466
min             6100.0
25%           104400.0
50%           196000.0
75%           331200.0
max          1717300.0
Name: bank_balance, dtype: Float64


In [9]:

import pandas as pd

df_clean = df.copy()

numeric_columns = ["age", "monthly_salary", "bank_balance"]

for col in numeric_columns:
    values = df_clean[col].astype("string").str.strip()

    # Repeatedly remove trailing .0 when it follows another .0
    while values.str.contains(r"\.0\.0$", regex=True, na=False).any():
        values = values.str.replace(r"(?<=\.0)\.0$", "", regex=True)

    # Normalize common textual representations of missing values
    values = values.mask(
        values.str.lower().isin(["nan", "nan.0", "none", "null", ""]),
        pd.NA
    )

    df_clean[col] = pd.to_numeric(values, errors="coerce")

print("Remaining missing values:")
print(df_clean[numeric_columns].isna().sum())

print("\nRemaining conversion failures:")
for col in numeric_columns:
    failed = df[col].notna() & df_clean[col].isna()
    print(f"{col}: {failed.sum()}")
    if failed.any():
        print(df.loc[failed, col].astype(str).value_counts().head(10))


Remaining missing values:
age                  0
monthly_salary       0
bank_balance      2440
dtype: int64

Remaining conversion failures:
age: 0
monthly_salary: 0
bank_balance: 14
bank_balance
nan.0    14
Name: count, dtype: int64


In [10]:

print("Dataset shape:", df_clean.shape)

print("\nMissing values across all columns:")
missing = df_clean.isna().sum()
print(missing[missing > 0].sort_values(ascending=False))

print("\nEligibility target distribution:")
print(df_clean["emi_eligibility"].value_counts(dropna=False))

print("\nMaximum monthly EMI summary:")
print(df_clean["max_monthly_emi"].describe())


Dataset shape: (404800, 27)

Missing values across all columns:
bank_balance      2440
monthly_rent      2426
credit_score      2420
education         2404
emergency_fund    2351
dtype: int64

Eligibility target distribution:
emi_eligibility
Not_Eligible    312868
Eligible         74444
High_Risk        17488
Name: count, dtype: int64

Maximum monthly EMI summary:
count    404800.000000
mean       6763.602156
std        7741.263317
min         500.000000
25%         500.000000
50%        4211.200000
75%        9792.000000
max       91040.400000
Name: max_monthly_emi, dtype: float64


In [11]:
print("Dataset shape:", df_clean.shape)

print("\nMissing values:")
missing = df_clean.isna().sum()
print(missing[missing > 0].sort_values(ascending=False))

print("\nDuplicate rows:", df_clean.duplicated().sum())

print("\nData types:")
print(df_clean.dtypes)

Dataset shape: (404800, 27)

Missing values:
bank_balance      2440
monthly_rent      2426
credit_score      2420
education         2404
emergency_fund    2351
dtype: int64



Duplicate rows: 0

Data types:
age                       Float64
gender                        str
marital_status                str
education                     str
monthly_salary            Float64
employment_type               str
years_of_employment       float64
company_type                  str
house_type                    str
monthly_rent              float64
family_size                 int64
dependents                  int64
school_fees               float64
college_fees              float64
travel_expenses           float64
groceries_utilities       float64
other_monthly_expenses    float64
existing_loans                str
current_emi_amount        float64
credit_score              float64
bank_balance              Float64
emergency_fund            float64
emi_scenario                  str
requested_amount          float64
requested_tenure            int64
emi_eligibility               str
max_monthly_emi           float64
dtype: object


In [12]:
print("EMI Eligibility Distribution:")
print(df_clean["emi_eligibility"].value_counts(dropna=False))

print("\nMaximum Monthly EMI Summary:")
print(df_clean["max_monthly_emi"].describe())

print("\nMissing target values:")
print(
    df_clean[["emi_eligibility", "max_monthly_emi"]]
    .isna()
    .sum()
)

EMI Eligibility Distribution:
emi_eligibility
Not_Eligible    312868
Eligible         74444
High_Risk        17488
Name: count, dtype: int64

Maximum Monthly EMI Summary:
count    404800.000000
mean       6763.602156
std        7741.263317
min         500.000000
25%         500.000000
50%        4211.200000
75%        9792.000000
max       91040.400000
Name: max_monthly_emi, dtype: float64

Missing target values:
emi_eligibility    0
max_monthly_emi    0
dtype: int64
